In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

### Load the csv

In [3]:
df_model = pd.read_csv("../data/processed/df_young_model_ready_2024.csv")

print(df_model.shape)
df_model.head()

(13376, 57)


,collision_year_x,vehicle_type,towing_and_articulation,vehicle_manoeuvre_historic,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,...,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,collision_injury_based,high_risk,hour,month
0,2024,9,0,-1,3,8,4,0,1,0,...,-1,-1,0,1,2,2,1,0,21,3
1,2024,9,0,-1,4,3,7,0,0,0,...,-1,-1,0,1,1,2,1,0,17,12
2,2024,9,0,18,19,7,3,0,4,0,...,0,0,0,1,1,-1,0,0,18,7
3,2024,9,0,18,19,3,6,0,0,5,...,0,0,0,2,1,1,0,0,8,6
4,2024,9,0,-1,7,3,6,0,0,0,...,-1,-1,0,1,3,1,1,1,22,2


### Remove Police Attendance Feature

In [4]:
df_model = df_model.drop(columns=["did_police_officer_attend_scene_of_accident"])

In [7]:
print("Police feature present?:", "did_police_officer_attend_scene_of_accident" in df_model.columns)

Police feature present?: False


### Define Target and Features

In [8]:
y = df_model["high_risk"].astype(int)

X = df_model.drop(columns=[
    "high_risk",
    "sex_of_driver",
    "age_band_of_driver"
])

### Train / Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (10700, 53)
Test: (2676, 53)


## Train Models

### Logistic Regression

In [10]:
lr = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

lr.fit(X_train, y_train)

y_proba_lr = lr.predict_proba(X_test)[:,1]

/Users/sinjinisarkar/Desktop/3rd Year - CWKs/Semester 2/Individual Project/Projects/fyp-insurance-fairness-xai/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Scaled version of LR

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

lr_pipeline.fit(X_train, y_train)

y_proba_lr = lr_pipeline.predict_proba(X_test)[:,1]

### Random Forest

In [13]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_proba_rf = rf.predict_proba(X_test)[:,1]

### XGBoost

In [14]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_proba_xgb = xgb.predict_proba(X_test)[:,1]

In [15]:
threshold_lr = 0.5
threshold_rf = 0.3
threshold_xgb = 0.3

In [16]:
y_pred_lr = (y_proba_lr >= threshold_lr).astype(int)
y_pred_rf = (y_proba_rf >= threshold_rf).astype(int)
y_pred_xgb = (y_proba_xgb >= threshold_xgb).astype(int)

In [17]:
def evaluate_model(y_true, y_pred, y_proba):

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)

    return accuracy, precision, recall, f1, auc

In [18]:
results = {}

results["Logistic Regression"] = evaluate_model(y_test, y_pred_lr, y_proba_lr)
results["Random Forest"] = evaluate_model(y_test, y_pred_rf, y_proba_rf)
results["XGBoost"] = evaluate_model(y_test, y_pred_xgb, y_proba_xgb)

In [19]:
comparison_no_police = pd.DataFrame(
    results,
    index=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
).T

comparison_no_police

,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.607623,0.357561,0.603667,0.449108,0.640660
Random Forest,0.663303,0.410781,0.623413,0.495238,0.699482
XGBoost,0.699552,0.444183,0.533145,0.484615,0.705509
